# Make ESM-embedding inputs for PST from the training and test data

In [44]:
! pip install pyfastatools tqdm --quiet

## Load and format pre-built training and test data protein tables

In [6]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [1]:
from pathlib import Path
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data")
SPLIT_DIR = ROOT_DIR.joinpath("train_test_splits")

In [47]:
! ls -lah {SPLIT_DIR}

total 6.2G
drwxrwxr-x 2 kosmopoulos kosmopoulos 4.0K Jul 11 14:22 .
drwxrwxr-x 5 kosmopoulos kosmopoulos 4.0K Jul 10 14:26 ..
-rw-rw-r-- 1 kosmopoulos kosmopoulos 109M Jul 11 14:31 test_equal_pos.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos  15M Jul 11 14:22 test_equal_pos.parquet
-rw-rw-r-- 1 kosmopoulos kosmopoulos  57M Jul 11 14:31 test_equal_source.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos 8.4M Jul 11 14:22 test_equal_source.parquet
-rw-rw-r-- 1 kosmopoulos kosmopoulos 178M Jul 11 14:31 test_half_virus_host.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos  24M Jul 11 14:22 test_half_virus_host.parquet
-rw-rw-r-- 1 kosmopoulos kosmopoulos  95M Jul 11 14:31 test_host_enriched.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos  14M Jul 11 14:22 test_host_enriched.parquet
-rw-rw-r-- 1 kosmopoulos kosmopoulos 278M Jul 11 14:31 test_input.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos  38M Jul 11 14:22 test_input.parquet
-rw-rw-r-- 1 kosmopoulos kosmopoulos  59M Jul 11 14:31 test_mge_enriched.faa
-rw-rw-r-- 1 kosmop

In [48]:
import glob

parquet_paths = glob.glob(os.path.join(SPLIT_DIR, "*.parquet"))

# sort so "train" is first, everything else alphabetical after
parquet_paths = sorted(
    parquet_paths,
    key=lambda p: (os.path.basename(p).replace(".parquet", "") != "train",
                   os.path.basename(p))
)

train_test_dfs = {}

for dataset_path in parquet_paths:
    dataset_name = os.path.basename(dataset_path).replace(".parquet", "")
    print(f"Loading {dataset_name} from {dataset_path}")
    train_test_dfs[dataset_name] = (
        pl.read_parquet(dataset_path)
        .sort(["Contig", "contig_pos_start", "contig_pos_end"])
    )

Loading train from /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/train.parquet


Loading test_equal_pos from /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_equal_pos.parquet
Loading test_equal_source from /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_equal_source.parquet
Loading test_half_virus_host from /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_half_virus_host.parquet
Loading test_host_enriched from /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_host_enriched.parquet
Loading test_input from /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_input.parquet
Loading test_mge_enriched from /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_mge_enriched.parquet
Loading test_near_all_host from /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_near_all_host.parquet
Loading test_near_all_virus from /storage2/scratch/

## Label AVG and viral positives and negatives
Positive viral proteins will be marked according to their curated database/sequence source, plus additional filtering for the progenomes dataset (same labels as the LGBM training data).

Positive AVGs will be proteins classified as metabolic, physiological, or regulatory by CheckAMG v1.1 based on their annotations. Importantly, they will **not** be curated according to genome context like CheckAMG annotate does by default. This means that runs of 3+ AVGs in a row will be allowed, as well as AVGs at the ends of contigs, and AVGs that are predicted or actually labeled as nonviral will all be allowed to be labeled as AVG-like. This so the CheckAMG-PST model can learn the features of an AVG-like *annotation* and use that to predict AVGs independently of its viral origin and independent of additional, conservation curation steps, which can be applied after the model. This is also necessary for how the CheckAMG-PST model will sample negatives when calculating model loss.

In [49]:
AVG_CLASSES = ["metabolic", "physiological", "regulatory"]
MAX_CONTIG_SIZE = 2048

In [50]:
def add_avg_label(df: pl.DataFrame) -> pl.DataFrame:
    df = (
        df.select([
            "Protein",
            "Contig",
            "frame",
            "gene_number",
            "True Positive",
            "Protein Classification",
            "contig_left_end_gene_dist",
            "contig_right_end_gene_dist",
        ])
        .rename({"True Positive": "viral"})
        .with_columns(
            pl.col("Protein Classification").is_in(AVG_CLASSES).alias("AVG")
        )
        .drop("Protein Classification")
        .sort(["Contig", "gene_number"])
    )

    return df

In [51]:
def print_basic_stats(df: pl.DataFrame, label: str) -> None:
    print(f"[{label}] proteins: {df.height:,}")
    print(f"[{label}] contigs:  {df.select(pl.col('Contig').n_unique()).item():,}")
    if "AVG" in df.columns:
        print(f"[{label}] AVG positives: {df.select(pl.col('AVG').sum()).item():,}")
    if "viral" in df.columns:
        print(f"[{label}] viral positives: {df.select(pl.col('viral').sum()).item():,}")
    print("")

In [52]:
train_test_dfs.keys()

dict_keys(['train', 'test_equal_pos', 'test_equal_source', 'test_half_virus_host', 'test_host_enriched', 'test_input', 'test_mge_enriched', 'test_near_all_host', 'test_near_all_virus', 'test_provirus', 'test_virus_enriched'])

In [53]:
labeled_dfs = {}
for name, df in train_test_dfs.items():
    labeled_dfs[name] = add_avg_label(df)
    print_basic_stats(labeled_dfs[name], name)

[train] proteins: 13,999,957
[train] contigs:  1,196,497
[train] AVG positives: 1,398,818
[train] viral positives: 5,279,901

[test_equal_pos] proteins: 383,948
[test_equal_pos] contigs:  45,115
[test_equal_pos] AVG positives: 30,021
[test_equal_pos] viral positives: 190,338

[test_equal_source] proteins: 191,832
[test_equal_source] contigs:  22,427
[test_equal_source] AVG positives: 18,444
[test_equal_source] viral positives: 62,666

[test_half_virus_host] proteins: 635,737
[test_half_virus_host] contigs:  71,335
[test_half_virus_host] AVG positives: 54,325
[test_half_virus_host] viral positives: 315,550

[test_host_enriched] proteins: 313,192
[test_host_enriched] contigs:  36,853
[test_host_enriched] AVG positives: 40,075
[test_host_enriched] viral positives: 53,037

[test_input] proteins: 960,240
[test_input] contigs:  114,890
[test_input] AVG positives: 92,519
[test_input] viral positives: 374,466

[test_mge_enriched] proteins: 203,839
[test_mge_enriched] contigs:  19,919
[test_mge

## Remove single-protein scaffolds/genomes
Since PST cannot handle these

In [54]:
def remove_single_protein_contigs(df: pl.DataFrame, label: str) -> pl.DataFrame:
    contigs_before = df.select(pl.col("Contig").n_unique()).item()
    prots_before = df.height

    df2 = (
        df.lazy()
        .join(
            df.lazy()
            .group_by("Contig")
            .agg(pl.len().alias("n_proteins")),
            on="Contig",
            how="inner",
        )
        .filter(pl.col("n_proteins") > 1)
        .drop("n_proteins")
        .collect()
    )

    contigs_after = df2.select(pl.col("Contig").n_unique()).item()
    prots_after = df2.height

    print(f"[{label}] Number of contigs before removing single-protein contigs: {contigs_before:,}")
    print(f"[{label}] Number of contigs after removing single-protein contigs: {contigs_after:,}")
    print(f"[{label}] Percent of contigs removed: {100 * (1 - contigs_after / contigs_before):.2f}%")
    print("")
    print(f"[{label}] Number of proteins before removing single-protein contigs: {prots_before:,}")
    print(f"[{label}] Number of proteins after removing single-protein contigs: {prots_after:,}")
    print(f"[{label}] Percent of proteins removed: {100 * (1 - prots_after / prots_before):.2f}%")
    print("")

    return df2

In [55]:
filtered_dfs = {}
for name, df in labeled_dfs.items():
    filtered_dfs[name] = remove_single_protein_contigs(df, name)

[train] Number of contigs before removing single-protein contigs: 1,196,497
[train] Number of contigs after removing single-protein contigs: 1,138,907
[train] Percent of contigs removed: 4.81%

[train] Number of proteins before removing single-protein contigs: 13,999,957
[train] Number of proteins after removing single-protein contigs: 13,942,367
[train] Percent of proteins removed: 0.41%

[test_equal_pos] Number of contigs before removing single-protein contigs: 45,115
[test_equal_pos] Number of contigs after removing single-protein contigs: 42,092
[test_equal_pos] Percent of contigs removed: 6.70%

[test_equal_pos] Number of proteins before removing single-protein contigs: 383,948
[test_equal_pos] Number of proteins after removing single-protein contigs: 380,925
[test_equal_pos] Percent of proteins removed: 0.79%

[test_equal_source] Number of contigs before removing single-protein contigs: 22,427
[test_equal_source] Number of contigs after removing single-protein contigs: 20,738
[te

## Split scaffolds with >2048 proteins into multiple
Since the max PST can handle per scaffold is 2048

In [56]:
import math

def split_large_contigs(
    df: pl.DataFrame,
    label: str,
    max_size: int = 2048,
    protein_prefix_delim: str = "|",
) -> tuple[pl.DataFrame, dict[str, str]]:
    df_tmp = df.with_row_index("_orig_idx")

    contig_counts = df_tmp.group_by("Contig").agg(pl.len().alias("n"))
    to_split = contig_counts.filter(pl.col("n") > max_size).select(["Contig", "n"])

    if to_split.height == 0:
        print(f"[{label}] No contigs > {max_size} proteins.")
        print("")
        return df_tmp.drop("_orig_idx"), {}

    print(f"[{label}] Contigs > {max_size} proteins: {to_split.height:,}")

    split_contigs = set(to_split.select("Contig").to_series().to_list())

    pieces: list[pl.DataFrame] = []
    ptn_rename_map: dict[str, str] = {}

    unchanged = df_tmp.filter(~pl.col("Contig").is_in(list(split_contigs)))
    pieces.append(unchanged)

    for contig, n in to_split.iter_rows():
        sub = df_tmp.filter(pl.col("Contig") == contig).sort("_orig_idx")
        n = sub.height

        k = math.ceil(n / max_size)
        base = n // k
        rem = n % k

        starts = []
        cur = 0
        for i in range(k):
            part_size = base + 1 if i < rem else base
            starts.append((cur, part_size))
            cur += part_size

        pad = len(str(len(starts)))

        for idx, (s, length) in enumerate(starts, start=1):
            chunk = sub.slice(s, length)
            part_suffix = f"_part{str(idx).zfill(pad)}"
            new_contig = contig + part_suffix

            old_proteins = chunk.select("Protein").to_series().to_list()

            def fix_protein(name: str, old_prefix=contig, new_prefix=new_contig, delim=protein_prefix_delim):
                if name.startswith(old_prefix):
                    return name.replace(old_prefix, new_prefix, 1)
                return new_prefix + delim + name

            new_proteins = [fix_protein(p) for p in old_proteins]
            for o, nn in zip(old_proteins, new_proteins):
                ptn_rename_map[o] = nn

            chunk = chunk.with_columns([
                pl.lit(new_contig).alias("Contig"),
                pl.Series("Protein", new_proteins),
            ])
            pieces.append(chunk)

    out = pl.concat(pieces).sort("_orig_idx").drop("_orig_idx")

    print(f"[{label}] Proteins after splitting: {out.height:,}")
    print(f"[{label}] Contigs after splitting:  {out.select(pl.col('Contig').n_unique()).item():,}")
    print("")

    return out, ptn_rename_map

In [57]:
split_dfs = {}
ptn_rename_maps = {}
for name, df in filtered_dfs.items():
    split_dfs[name], ptn_rename_maps[name] = split_large_contigs(df, label=name, max_size=MAX_CONTIG_SIZE)

[train] Contigs > 2048 proteins: 642
[train] Proteins after splitting: 13,942,367
[train] Contigs after splitting:  1,139,814

[test_equal_pos] No contigs > 2048 proteins.

[test_equal_source] No contigs > 2048 proteins.

[test_half_virus_host] No contigs > 2048 proteins.

[test_host_enriched] No contigs > 2048 proteins.

[test_input] No contigs > 2048 proteins.

[test_mge_enriched] No contigs > 2048 proteins.

[test_near_all_host] No contigs > 2048 proteins.

[test_near_all_virus] No contigs > 2048 proteins.

[test_provirus] No contigs > 2048 proteins.

[test_virus_enriched] No contigs > 2048 proteins.



In [58]:
def check_splits(df: pl.DataFrame, label: str, max_size: int = 2048, imbalance_tol: float = 0.20) -> pl.DataFrame:
    counts = df.group_by("Contig").agg(pl.len().alias("n"))
    big = counts.filter(pl.col("n") > max_size)
    if big.height == 0:
        print(f"[{label}] No contigs exceed max_size={max_size}.")
        print("")
        return pl.DataFrame([])

    parts = (
        df.select("Contig")
        .unique()
        .with_columns([
            pl.col("Contig").str.extract(r"^(.*)_part\d+$", 1).fill_null(pl.col("Contig")).alias("orig_contig"),
            pl.col("Contig").str.extract(r"_part(\d+)$", 1).cast(pl.Int64).alias("part_idx"),
        ])
    )

    chunk_sizes = (
        df.join(parts, on="Contig", how="left")
        .group_by(["orig_contig", "part_idx"])
        .agg(pl.len().alias("chunk_n"))
        .sort(["orig_contig", "part_idx"])
    )

    reports = []
    for orig_contig in big.select("Contig").to_series().to_list():
        sub = chunk_sizes.filter(pl.col("orig_contig") == orig_contig).sort("part_idx")
        if sub.height == 0:
            continue
        sizes = sub.select("chunk_n").to_series().to_list()
        mean_chunk = sum(sizes) / len(sizes)
        variance = sum((x - mean_chunk) ** 2 for x in sizes) / len(sizes)
        std = math.sqrt(variance)
        cv = std / mean_chunk if mean_chunk else 0.0
        imbalanced = (cv > imbalance_tol) or ((max(sizes) - min(sizes)) / mean_chunk > imbalance_tol)

        reports.append({
            "contig": orig_contig,
            "n_parts": int(len(sizes)),
            "min_chunk": int(min(sizes)),
            "max_chunk": int(max(sizes)),
            "mean_chunk": float(mean_chunk),
            "cv": float(cv),
            "imbalanced": bool(imbalanced),
        })

    n_contigs = len(reports)
    n_imbalanced = sum(1 for r in reports if r["imbalanced"])
    print(f"[{label}] Checked {n_contigs} contigs > {max_size}.")
    print(f"[{label}] Contigs with imbalanced chunk sizes (tol={imbalance_tol*100:.0f}%): {n_imbalanced}")
    print("")

    return pl.from_dicts(reports)

In [59]:
for name, df in split_dfs.items():
    report = check_splits(df, label=name, max_size=MAX_CONTIG_SIZE, imbalance_tol=0.20)
    if report.height > 0:
        print(f"[{name}] Contigs with imbalanced splits:")
        print(report.filter(pl.col("imbalanced") == True))

[train] No contigs exceed max_size=2048.

[test_equal_pos] No contigs exceed max_size=2048.

[test_equal_source] No contigs exceed max_size=2048.

[test_half_virus_host] No contigs exceed max_size=2048.

[test_host_enriched] No contigs exceed max_size=2048.

[test_input] No contigs exceed max_size=2048.

[test_mge_enriched] No contigs exceed max_size=2048.

[test_near_all_host] No contigs exceed max_size=2048.

[test_near_all_virus] No contigs exceed max_size=2048.

[test_provirus] No contigs exceed max_size=2048.

[test_virus_enriched] No contigs exceed max_size=2048.



## Map renamed proteins

In [60]:
from collections import Counter

def report_rename_map_health(name: str, df_filtered: pl.DataFrame, rename_map: dict[str, str], max_show: int = 5) -> None:
    keys = set(rename_map.keys())
    vals = list(rename_map.values())

    n_rows = df_filtered.height
    n_in = df_filtered.select(pl.col("Protein").is_in(list(keys)).sum()).item()
    print(f"[{name}] filtered rows: {n_rows:,}")
    print(f"[{name}] filtered proteins covered by map keys: {n_in:,} ({100*n_in/n_rows:.2f}%)")

    c = Counter(vals)
    n_collisions = sum(v > 1 for v in c.values())
    print(f"[{name}] rename_map size: {len(rename_map):,}")
    print(f"[{name}] unique new ids: {len(c):,}")
    print(f"[{name}] collisions (new id reused): {n_collisions:,}")
    if n_collisions:
        print(f"[{name}] collision examples:")
        shown = 0
        for new_id, ct in c.items():
            if ct > 1:
                print(f"  {new_id} -> {ct} old ids")
                shown += 1
                if shown >= max_show:
                    break
    print("")

for name in filtered_dfs:
    report_rename_map_health(name, filtered_dfs[name], ptn_rename_maps.get(name, {}))

[train] filtered rows: 13,942,367
[train] filtered proteins covered by map keys: 2,407,878 (17.27%)
[train] rename_map size: 2,407,878
[train] unique new ids: 2,407,878
[train] collisions (new id reused): 0

[test_equal_pos] filtered rows: 380,925
[test_equal_pos] filtered proteins covered by map keys: 0 (0.00%)
[test_equal_pos] rename_map size: 0
[test_equal_pos] unique new ids: 0
[test_equal_pos] collisions (new id reused): 0

[test_equal_source] filtered rows: 190,143
[test_equal_source] filtered proteins covered by map keys: 0 (0.00%)
[test_equal_source] rename_map size: 0
[test_equal_source] unique new ids: 0
[test_equal_source] collisions (new id reused): 0

[test_half_virus_host] filtered rows: 632,631
[test_half_virus_host] filtered proteins covered by map keys: 0 (0.00%)
[test_half_virus_host] rename_map size: 0
[test_half_virus_host] unique new ids: 0
[test_half_virus_host] collisions (new id reused): 0

[test_host_enriched] filtered rows: 311,522
[test_host_enriched] filtere

## Subset and format columns to just those needed

In [61]:
final_dfs = {}
for name, df in split_dfs.items():
    # fail fast if Protein IDs are not unique at this stage
    has_dups = df.select(pl.col("Protein").is_duplicated().any()).item()
    if has_dups:
        raise RuntimeError(f"[{name}] duplicate Protein IDs detected in split_dfs; fix split naming before continuing")

    final_dfs[name] = (
        df
        .sort(["Contig", "gene_number"])
        .select(["Contig", "Protein", "frame", "gene_number", "viral", "AVG"])
    )

    print(f"[{name}] Final proteins: {final_dfs[name].height:,}")
    print(final_dfs[name].get_column("viral").value_counts())
    print(final_dfs[name].get_column("AVG").value_counts())
    print("")

[train] Final proteins: 13,942,367
shape: (2, 2)
┌───────┬─────────┐
│ viral ┆ count   │
│ ---   ┆ ---     │
│ bool  ┆ u32     │
╞═══════╪═════════╡
│ true  ┆ 5242078 │
│ false ┆ 8700289 │
└───────┴─────────┘
shape: (2, 2)
┌───────┬──────────┐
│ AVG   ┆ count    │
│ ---   ┆ ---      │
│ bool  ┆ u32      │
╞═══════╪══════════╡
│ false ┆ 12545137 │
│ true  ┆ 1397230  │
└───────┴──────────┘

[test_equal_pos] Final proteins: 380,925
shape: (2, 2)
┌───────┬────────┐
│ viral ┆ count  │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ false ┆ 192629 │
│ true  ┆ 188296 │
└───────┴────────┘
shape: (2, 2)
┌───────┬────────┐
│ AVG   ┆ count  │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ false ┆ 351013 │
│ true  ┆ 29912  │
└───────┴────────┘

[test_equal_source] Final proteins: 190,143
shape: (2, 2)
┌───────┬────────┐
│ viral ┆ count  │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ true  ┆ 61514  │
│ false ┆ 128629 │
└───────┴────────┘
shape: (2, 2)
┌───────┬─

In [62]:
def report_dups(df: pl.DataFrame, name: str, n_show: int = 20) -> pl.DataFrame:
    dup_counts = (
        df
        .group_by("Protein")
        .agg(pl.len().alias("n"))
        .filter(pl.col("n") > 1)
        .sort("n", descending=True)
    )

    n_dup_ids = dup_counts.height
    n_dup_rows = dup_counts.select(pl.col("n").sum()).item() if n_dup_ids else 0

    print(f"[{name}] duplicate Protein IDs: {n_dup_ids:,}")
    print(f"[{name}] total rows in duplicated IDs: {n_dup_rows:,}")
    if n_dup_ids:
        print(dup_counts.head(n_show))
    print("")
    return dup_counts

dup_reports = {}
for name, df in split_dfs.items():
    dup_reports[name] = report_dups(df, name)

[train] duplicate Protein IDs: 0
[train] total rows in duplicated IDs: 0

[test_equal_pos] duplicate Protein IDs: 0
[test_equal_pos] total rows in duplicated IDs: 0

[test_equal_source] duplicate Protein IDs: 0
[test_equal_source] total rows in duplicated IDs: 0

[test_half_virus_host] duplicate Protein IDs: 0
[test_half_virus_host] total rows in duplicated IDs: 0

[test_host_enriched] duplicate Protein IDs: 0
[test_host_enriched] total rows in duplicated IDs: 0

[test_input] duplicate Protein IDs: 0
[test_input] total rows in duplicated IDs: 0

[test_mge_enriched] duplicate Protein IDs: 0
[test_mge_enriched] total rows in duplicated IDs: 0

[test_near_all_host] duplicate Protein IDs: 0
[test_near_all_host] total rows in duplicated IDs: 0

[test_near_all_virus] duplicate Protein IDs: 0
[test_near_all_virus] total rows in duplicated IDs: 0

[test_provirus] duplicate Protein IDs: 0
[test_provirus] total rows in duplicated IDs: 0

[test_virus_enriched] duplicate Protein IDs: 0
[test_virus

## Get the fraction of contigs with pos. and neg. viral and AVGs

In [63]:
def summarize_contigs(df: pl.DataFrame, contig_col: str, bool_cols: list[str]) -> pl.DataFrame:
    contig_summary = (
        df.group_by(contig_col)
        .agg([
            *(pl.any(c).alias(f"has_{c}") for c in bool_cols),
            pl.len().alias("n_proteins")
        ])
    )

    n_total_contigs = contig_summary.height
    n_total_proteins = contig_summary["n_proteins"].sum()

    summaries = []
    for c in bool_cols:
        n_with = contig_summary[f"has_{c}"].sum()
        n_without = n_total_contigs - n_with

        n_prots_with = (
            contig_summary
            .filter(pl.col(f"has_{c}"))
            ["n_proteins"].sum()
        )
        n_prots_without = n_total_proteins - n_prots_with

        summaries.append({
            "feature": c,
            "n_contigs_with": n_with,
            "frac_contigs_with": n_with / n_total_contigs,
            "n_contigs_without": n_without,
            "frac_contigs_without": n_without / n_total_contigs,
            "n_proteins_on_contigs_with": n_prots_with,
            "frac_proteins_on_contigs_with": n_prots_with / n_total_proteins,
            "n_proteins_on_contigs_without": n_prots_without,
            "frac_proteins_on_contigs_without": n_prots_without / n_total_proteins,
        })

    return pl.DataFrame(summaries)

In [64]:
for name, df in final_dfs.items():
    with pl.Config(tbl_cols=-1):
        print(f"Summary for {name}:")
        print(summarize_contigs(df, contig_col="Contig", bool_cols=["viral", "AVG"]))
        print("")

Summary for train:
shape: (2, 9)
┌─────────┬───────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ feature ┆ n_contigs ┆ frac_con ┆ n_contig ┆ frac_con ┆ n_protei ┆ frac_pro ┆ n_protei ┆ frac_pro │
│ ---     ┆ _with     ┆ tigs_wit ┆ s_withou ┆ tigs_wit ┆ ns_on_co ┆ teins_on ┆ ns_on_co ┆ teins_on │
│ str     ┆ ---       ┆ h        ┆ t        ┆ hout     ┆ ntigs_wi ┆ _contigs ┆ ntigs_wi ┆ _contigs │
│         ┆ i64       ┆ ---      ┆ ---      ┆ ---      ┆ th       ┆ _with    ┆ thout    ┆ _without │
│         ┆           ┆ f64      ┆ i64      ┆ f64      ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│         ┆           ┆          ┆          ┆          ┆ i64      ┆ f64      ┆ i64      ┆ f64      │
╞═════════╪═══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ viral   ┆ 506967    ┆ 0.44478  ┆ 632847   ┆ 0.55522  ┆ 6434875  ┆ 0.461534 ┆ 7507492  ┆ 0.538466 │
│ AVG     ┆ 413662    ┆ 0.362921 ┆ 726152   ┆ 0.637079 ┆ 8

## Write the PST train and test data tables

In [2]:
PST_OUT = ROOT_DIR.parent.joinpath("pst/training")
TRAIN_OUT = PST_OUT.joinpath("train_data")
TEST_OUT = PST_OUT.joinpath("test_data")

In [66]:
os.makedirs(TRAIN_OUT, exist_ok=True)
os.makedirs(TEST_OUT, exist_ok=True)

In [67]:
final_dfs.keys()

dict_keys(['train', 'test_equal_pos', 'test_equal_source', 'test_half_virus_host', 'test_host_enriched', 'test_input', 'test_mge_enriched', 'test_near_all_host', 'test_near_all_virus', 'test_provirus', 'test_virus_enriched'])

In [68]:
for name, df in final_dfs.items():
    if name == "train" or name.startswith("train_") or "train" in name:
        out_path = TRAIN_OUT.joinpath(f"{name}_ptns.parquet")
    else:
        out_path = TEST_OUT.joinpath(f"{name}_ptns.parquet")

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.write_parquet(out_path)

    n_rows = df.height if hasattr(df, "height") else len(df)
    n_unique_prot = df.select(pl.col("Protein").n_unique()).item() if "Protein" in df.columns else None

    if n_unique_prot is None:
        print(f"[{name}] wrote {n_rows:,} rows -> {out_path}")
    else:
        print(f"[{name}] wrote {n_rows:,} rows (unique proteins={n_unique_prot:,}) -> {out_path}")

[train] wrote 13,942,367 rows (unique proteins=13,942,367) -> /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_ptns.parquet
[test_equal_pos] wrote 380,925 rows (unique proteins=380,925) -> /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_pos_ptns.parquet
[test_equal_source] wrote 190,143 rows (unique proteins=190,143) -> /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_source_ptns.parquet
[test_half_virus_host] wrote 632,631 rows (unique proteins=632,631) -> /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_half_virus_host_ptns.parquet
[test_host_enriched] wrote 311,522 rows (unique proteins=311,522) -> /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_host_enriched_ptns.parquet
[test_input] wrote 953,918 rows (unique proteins=953,918) -> /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_input_ptns.parquet
[test_mge

## Ensure the protein fastas for the train and test data are the same order as they appear in the tables
Also rename the proteins that were on split scaffolds so they match their new scaffold names.

In [69]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [70]:
from pyfastatools import Parser

def write_ordered_fasta_from_final(
    fasta_in: str,
    fasta_out: str,
    ordered_new_ids: list[str],
    old_to_new: dict[str, str],
    label: str,
) -> list[str]:
    # Build inverse map only for renamed proteins (new -> old)
    # old_to_new is expected to contain ONLY the changed ids (split contigs), not identity mappings.
    new_to_old = {}
    for old, new in old_to_new.items():
        if new in new_to_old:
            raise RuntimeError(f"[{label}] collision in rename map (new id not unique): {new}")
        new_to_old[new] = old

    # For proteins not in new_to_old, old id == new id (identity mapping)
    needed_old = [new_to_old.get(new_id, new_id) for new_id in ordered_new_ids]
    needed_old_set = set(needed_old)

    # Load sequences for all needed old ids
    seqs_old = {}
    n_seen = 0
    for record in Parser(fasta_in):
        old = record.header.name
        if old in needed_old_set:
            seqs_old[old] = record
            n_seen += 1

    print(f"[{label}] input fasta:  {fasta_in}")
    print(f"[{label}] output fasta: {fasta_out}")
    print(f"[{label}] proteins (final table): {len(ordered_new_ids):,}")
    print(f"[{label}] rename_map size (changed only): {len(old_to_new):,}")
    print(f"[{label}] sequences loaded from fasta: {n_seen:,} (unique loaded={len(seqs_old):,})")

    Path(fasta_out).parent.mkdir(parents=True, exist_ok=True)

    n_written = 0
    missing_seq = 0

    with open(fasta_out, "w") as out:
        for new_id in ordered_new_ids:
            old_id = new_to_old.get(new_id, new_id)  # identity if not renamed
            rec = seqs_old.get(old_id)
            if rec is None:
                missing_seq += 1
                continue
            out.write(f">{new_id}\n{rec.seq}\n")
            n_written += 1

    print(f"[{label}] sequences written: {n_written:,}")
    print(f"[{label}] missing due to no sequence in input fasta: {missing_seq:,}")
    print("")

    return ordered_new_ids

def process_one_split(
    name: str,
    df_final: pl.DataFrame,
    old_to_new: dict[str, str],
    fasta_in_dir: str,
    fasta_out_dir: str,
) -> tuple[str, list[str]]:
    fasta_in = os.path.join(fasta_in_dir, f"{name}.faa")
    fasta_out = os.path.join(fasta_out_dir, f"{name}_ptns.faa")

    ordered_new_ids = df_final["Protein"].to_list()

    ordered_new_ids = write_ordered_fasta_from_final(
        fasta_in=fasta_in,
        fasta_out=fasta_out,
        ordered_new_ids=ordered_new_ids,
        old_to_new=old_to_new,
        label=name,
    )
    return fasta_out, ordered_new_ids

In [71]:
expected_orders = {}

names = list(final_dfs.keys())
names = ["train"] + sorted([n for n in names if n != "train"])

for name in names:
    df = final_dfs[name]

    if name == "train":
        out_dir = TRAIN_OUT
    else:
        out_dir = TEST_OUT

    fasta_out, ordered_new_ids = process_one_split(
        name=name,
        df_final=df,
        old_to_new=ptn_rename_maps.get(name, {}),
        fasta_in_dir=SPLIT_DIR,
        fasta_out_dir=out_dir,
    )

    expected_orders[name] = {
        "fasta_out": fasta_out,
        "expected_ids": ordered_new_ids,
    }

[train] input fasta:  /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/train.faa
[train] output fasta: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_ptns.faa
[train] proteins (final table): 13,942,367
[train] rename_map size (changed only): 2,407,878
[train] sequences loaded from fasta: 13,942,367 (unique loaded=13,942,367)
[train] sequences written: 13,942,367
[train] missing due to no sequence in input fasta: 0

[test_equal_pos] input fasta:  /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_equal_pos.faa
[test_equal_pos] output fasta: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_pos_ptns.faa
[test_equal_pos] proteins (final table): 380,925
[test_equal_pos] rename_map size (changed only): 0
[test_equal_pos] sequences loaded from fasta: 380,925 (unique loaded=380,925)
[test_equal_pos] sequences written: 380,925
[test_equal_pos] missing due to no sequen

### Verify that it worked as intended

In [72]:
def verify_fasta_order(name: str, fasta_out: str, expected_ids: list[str], max_show: int = 10) -> bool:
    output_ids = [record.header.name for record in Parser(fasta_out)]

    if output_ids == expected_ids:
        print(f"[{name}] Passed: FASTA order matches expected list.")
        print(f"[{name}] n={len(output_ids):,}")
        print("")
        return True

    print(f"[{name}] Failed: FASTA order does not match expected list.")
    print(f"[{name}] expected n={len(expected_ids):,} output n={len(output_ids):,}")
    print("")

    n_min = min(len(expected_ids), len(output_ids))
    mismatches = [(i, expected_ids[i], output_ids[i]) for i in range(n_min) if expected_ids[i] != output_ids[i]]

    print(f"[{name}] mismatches (within overlap): {len(mismatches):,}")
    if mismatches:
        print(f"[{name}] first {min(max_show, len(mismatches))} mismatches (idx, expected, actual):")
        for idx, exp, act in mismatches[:max_show]:
            print(f"[{name}] {idx}: {exp} != {act}")

    if len(expected_ids) != len(output_ids):
        if len(expected_ids) > len(output_ids):
            print(f"[{name}] output is missing {len(expected_ids) - len(output_ids):,} ids at the end (or earlier).")
        else:
            print(f"[{name}] output has {len(output_ids) - len(expected_ids):,} extra ids (unexpected).")
    print("")
    return False

In [73]:
results = {}

names = list(expected_orders.keys())
names = ["train"] + sorted([n for n in names if n != "train"])

for name in names:
    info = expected_orders[name]
    results[name] = verify_fasta_order(
        name=name,
        fasta_out=info["fasta_out"],
        expected_ids=info["expected_ids"],
        max_show=10,
    )

[train] Passed: FASTA order matches expected list.
[train] n=13,942,367

[test_equal_pos] Passed: FASTA order matches expected list.
[test_equal_pos] n=380,925

[test_equal_source] Passed: FASTA order matches expected list.
[test_equal_source] n=190,143

[test_half_virus_host] Passed: FASTA order matches expected list.
[test_half_virus_host] n=632,631

[test_host_enriched] Passed: FASTA order matches expected list.
[test_host_enriched] n=311,522

[test_input] Passed: FASTA order matches expected list.
[test_input] n=953,918

[test_mge_enriched] Passed: FASTA order matches expected list.
[test_mge_enriched] n=203,361

[test_near_all_host] Passed: FASTA order matches expected list.
[test_near_all_host] n=665,307

[test_near_all_virus] Passed: FASTA order matches expected list.
[test_near_all_virus] n=841,115

[test_provirus] Passed: FASTA order matches expected list.
[test_provirus] n=75,260

[test_virus_enriched] Passed: FASTA order matches expected list.
[test_virus_enriched] n=425,458

In [74]:
import subprocess

def sh(cmd: str) -> str:
    return subprocess.check_output(cmd, shell=True, text=True).rstrip("\n")

def inspect_fasta(fasta_path: str, name: str, n: int = 5) -> None:
    print(f"[{name}] {fasta_path}")
    print(f"[{name}] head:")
    print(sh(f"head -n {n} {fasta_path}"))
    print("")
    print(f"[{name}] tail:")
    print(sh(f"tail -n {n} {fasta_path}"))
    print("")
    print(f"[{name}] n_records:")
    print(sh(f"grep -c '^>' {fasta_path}"))
    print("")

In [75]:
names = list(expected_orders.keys())
names = ["train"] + sorted([n for n in names if n != "train"])

for name in names:
    info = expected_orders[name]
    inspect_fasta(info["fasta_out"], name=name, n=10)

[train] /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_ptns.faa
[train] head:
>1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_1
MTYSQEEQAYFMTEALNEAKKSLEKNEIPIGCVIVKDGQIIGRGHNAREERQQAVMHAEIMAINEANAHEGNWRLLETTLFVTIEPCVMCSGAIGLARIPRVIFGAANPKFGGATSLYEILTDERLNHRVQVESGLLAKECAQMMQTFFRQRRDQKKEAKQEALTQTDSTPYQ
>1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_2
MKLTTLGSWGAYPHQDAGTTSYLITGCDGFQLLMDAGSRALNELEKEINPLDLDAVIISHYHPDHVADLGVLRHYYQLYPKHLKQAKCLPIYGHQEDPHEFAKLTIPQVSKGIAYQVDGVESIGPFDIRFIKTVHPVVCYAFRIEERQTGQVLVFTGDTGYFEGLIDFAKGADLFLADVYLYEGNENHIAHLTSKEAGQIAKKAGVKRLVLTHMPPLPPEGIDPENHLEVLRQETKTYAQDIPVELSLPHNSWDLGSDVR
>1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_3
MFSTKDLVRVAMMTSLIIILGFIPAIPLSFIPVPIVLQNLGIMLAAVLLGGKKGSLAVFLFLVVGLFLPVFSATKTTIPVLMGPTAGYVLAYPLVPLVFSLLYRKWLSQHTMMTFLAIFISGVLVVDVLGAIWLATYTGMPLGKSLLSNAVFIPGDTIKAIVATVIAVSYKDSFLNTKS
>1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_4
MAECFGIACDQNYRCQHYHSELDIVGLKCAAC

## Split training data into smaller files for more efficient processing
Also ensure no protein sequences are > 20k AAs, if they are, they will be fragmented. And, need to split scaffolds > 2048 proteins into multiple, since the max PST can handle is 2048 per scaffold.

In [76]:
from pyfastatools import Parser, write_fasta, Header, Record
from pathlib import Path
import os
from collections import defaultdict

from more_itertools import divide
from math import ceil

from tqdm import tqdm

In [77]:
MAX_SEQ_LIMIT = 20_000
TRAIN_N_FILES = 15

In [78]:
def split_to_fasta_path(split_name: str, parent_dir: Path) -> Path:
    if split_name == "train":
        return TRAIN_OUT / "train_ptns.faa"
    return TEST_OUT / f"{split_name}_ptns.faa"

split_names = list(final_dfs.keys())
split_names = ["train"] + sorted([k for k in split_names if k != "train"])

split_fasta_paths = {name: split_to_fasta_path(name, PST_OUT) for name in split_names}

for name, p in split_fasta_paths.items():
    print(f"{name}: {p}")

train: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_ptns.faa
test_equal_pos: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_pos_ptns.faa
test_equal_source: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_source_ptns.faa
test_half_virus_host: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_half_virus_host_ptns.faa
test_host_enriched: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_host_enriched_ptns.faa
test_input: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_input_ptns.faa
test_mge_enriched: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_mge_enriched_ptns.faa
test_near_all_host: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_near_all_host_ptns.faa
test_near_all_virus: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_

In [79]:
missing = [name for name, p in split_fasta_paths.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing FASTA(s): {missing}")
else:
    print(f"All FASTAs present: {len(split_fasta_paths):,}")

All FASTAs present: 11


In [80]:
counts = defaultdict(int)

for name, fasta_path in split_fasta_paths.items():
    for record in Parser(str(fasta_path)):
        if len(record.seq) >= MAX_SEQ_LIMIT:
            counts[name] += 1

counts

defaultdict(int, {'train': 1})

In [81]:
def iter_fragmented_records(records, max_len: int):
    for record in records:
        seqlen = len(record.seq)
        n_frags = ceil(seqlen / max_len)

        if n_frags > 1:
            name = record.header.name
            desc = record.header.desc

            for idx in range(n_frags):
                start = idx * max_len
                end = (idx + 1) * max_len
                seq = record.seq[start:end]
                header = Header(f"{name}_FRAGMENT_{idx}", desc)
                yield Record(header, seq) # type: ignore
        else:
            yield record

### Split the training data into chunks

In [82]:
TRAIN_IN = split_fasta_paths["train"]
TRAIN_WDIR = TRAIN_IN.parent
TRAIN_SPLIT_DIR = TRAIN_WDIR / "train_split"
TRAIN_SPLIT_DIR.mkdir(exist_ok=True)

In [83]:
records = list(Parser(str(TRAIN_IN)))
print(f"[train] n_records: {len(records):,}")
print(f"[train] max seqlen: {max(len(r.seq) for r in records):,}")

chunks = divide(n=TRAIN_N_FILES, iterable=records)
print(f"[train] n_chunks: {TRAIN_N_FILES:,}")

pbar = tqdm(total=len(records))
try:
    for i, chunk_iter in enumerate(chunks):
        pbar.set_description(f"train chunk {i + 1}/{TRAIN_N_FILES}")
        output = TRAIN_SPLIT_DIR / f"train_chunk_{i}.faa"

        n_in_chunk = 0
        with output.open("w") as fp:
            for rec in chunk_iter:
                n_in_chunk += 1
                for out_rec in iter_fragmented_records([rec], MAX_SEQ_LIMIT):
                    write_fasta(out_rec, fp)
                pbar.update(1)

        print(f"[train] wrote {output.name} (input records: {n_in_chunk:,})")
finally:
    pbar.close()

print(f"[train] wrote chunks to: {TRAIN_SPLIT_DIR}")

[train] n_records: 13,942,367
[train] max seqlen: 22,598
[train] n_chunks: 15


train chunk 2/15:   7%|▋         | 931887/13942367 [00:41<10:10, 21294.50it/s]

[train] wrote train_chunk_0.faa (input records: 929,492)


train chunk 3/15:  13%|█▎        | 1863180/13942367 [01:23<09:12, 21875.95it/s]

[train] wrote train_chunk_1.faa (input records: 929,492)


...

[train] wrote train_chunk_12.faa (input records: 929,491)


train chunk 15/15:  93%|█████████▎| 13018952/13942367 [08:30<00:28, 32097.53it/s]

[train] wrote train_chunk_13.faa (input records: 929,491)


train chunk 15/15: 100%|██████████| 13942367/13942367 [09:03<00:00, 25668.65it/s]

[train] wrote train_chunk_14.faa (input records: 929,491)
[train] wrote chunks to: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_split


Check the chunks:

In [84]:
chunk_paths = sorted(TRAIN_SPLIT_DIR.glob("train_chunk_*.faa"))
print(f"[train] chunk files: {len(chunk_paths):,}")

chunk_counts = {}
chunk_maxlen = {}

for p in chunk_paths:
    n = 0
    mx = 0
    for r in Parser(str(p)):
        n += 1
        mx = max(mx, len(r.seq))
    chunk_counts[p.name] = n
    chunk_maxlen[p.name] = mx

print("[train] chunk record counts (after fragmentation):")
list(chunk_counts.items())[:5], "..."

print("[train] max sequence length per chunk:")
max(chunk_maxlen.values()), "max overall"

[train] chunk files: 15
[train] chunk record counts (after fragmentation):
[train] max sequence length per chunk:


(20000, 'max overall')

### Test datasets

In [85]:
def rewrite_with_fragmentation_inplace(fasta_path: Path, max_len: int, label: str) -> None:
    records = list(Parser(str(fasta_path)))
    maxlen = max(len(r.seq) for r in records) if records else 0

    print(f"[{label}] n_records before: {len(records):,}")
    print(f"[{label}] max seqlen before: {maxlen:,}")

    out_tmp = fasta_path.with_suffix(fasta_path.suffix + ".tmp")
    n_out = 0
    mx_out = 0

    with out_tmp.open("w") as fp:
        for rec in iter_fragmented_records(records, max_len):
            mx_out = max(mx_out, len(rec.seq))
            write_fasta(rec, fp)
            n_out += 1

    out_tmp.replace(fasta_path)

    print(f"[{label}] n_records after: {n_out:,}")
    print(f"[{label}] max seqlen after: {mx_out:,}")
    print("")

In [86]:
for name in split_names:
    if name == "train":
        continue
    rewrite_with_fragmentation_inplace(split_fasta_paths[name], MAX_SEQ_LIMIT, label=name)

[test_equal_pos] n_records before: 380,925
[test_equal_pos] max seqlen before: 14,955
[test_equal_pos] n_records after: 380,925
[test_equal_pos] max seqlen after: 14,955

[test_equal_source] n_records before: 190,143
[test_equal_source] max seqlen before: 6,950
[test_equal_source] n_records after: 190,143
[test_equal_source] max seqlen after: 6,950

[test_half_virus_host] n_records before: 632,631
[test_half_virus_host] max seqlen before: 10,990
[test_half_virus_host] n_records after: 632,631
[test_half_virus_host] max seqlen after: 10,990

[test_host_enriched] n_records before: 311,522
[test_host_enriched] max seqlen before: 9,581
[test_host_enriched] n_records after: 311,522
[test_host_enriched] max seqlen after: 9,581

[test_input] n_records before: 953,918
[test_input] max seqlen before: 10,990
[test_input] n_records after: 953,918
[test_input] max seqlen after: 10,990

[test_mge_enriched] n_records before: 203,361
[test_mge_enriched] max seqlen before: 14,955
[test_mge_enriched] n

Check the test datasets:

In [87]:
def expected_new_ids_for_split(name: str) -> list[str]:
    return final_dfs[name]["Protein"].to_list()

def verify_fasta_order(name: str, fasta_path: Path, expected_ids: list[str], max_show: int = 10) -> bool:
    output_ids = [record.header.name for record in Parser(str(fasta_path))]

    if output_ids == expected_ids:
        print(f"[{name}] Passed: FASTA order matches expected list (n={len(output_ids):,}).")
        return True

    print(f"[{name}] Failed: FASTA order mismatch.")
    print(f"[{name}] expected n={len(expected_ids):,} output n={len(output_ids):,}")

    n_min = min(len(expected_ids), len(output_ids))
    mismatches = []
    for i in range(n_min):
        if expected_ids[i] != output_ids[i]:
            mismatches.append((i, expected_ids[i], output_ids[i]))
            if len(mismatches) >= max_show:
                break

    print(f"[{name}] first mismatches (index, expected, actual):")
    for i, e, a in mismatches:
        print(f"  {i}: {e} != {a}")
    print("")
    return False

In [88]:
results = {}
for name in split_names:
    exp = expected_new_ids_for_split(name)
    results[name] = verify_fasta_order(name, split_fasta_paths[name], exp, max_show=10)

print(f"All passed: {all(results.values())}")

[train] Passed: FASTA order matches expected list (n=13,942,367).
[test_equal_pos] Passed: FASTA order matches expected list (n=380,925).
[test_equal_source] Passed: FASTA order matches expected list (n=190,143).
[test_half_virus_host] Passed: FASTA order matches expected list (n=632,631).
[test_host_enriched] Passed: FASTA order matches expected list (n=311,522).
[test_input] Passed: FASTA order matches expected list (n=953,918).
[test_mge_enriched] Passed: FASTA order matches expected list (n=203,361).
[test_near_all_host] Passed: FASTA order matches expected list (n=665,307).
[test_near_all_virus] Passed: FASTA order matches expected list (n=841,115).
[test_provirus] Passed: FASTA order matches expected list (n=75,260).
[test_virus_enriched] Passed: FASTA order matches expected list (n=425,458).
All passed: True


### Inspect outputs

In [89]:
def inspect_fasta(fasta_path: Path, name: str, n: int = 5) -> None:
    headers = []
    n_records = 0
    last_headers = []

    for record in Parser(str(fasta_path)):
        n_records += 1
        if len(headers) < n:
            headers.append(record.header.name)
        last_headers.append(record.header.name)
        if len(last_headers) > n:
            last_headers.pop(0)

    print(f"[{name}] {fasta_path}")
    print(f"[{name}] n_records: {n_records:,}")
    print(f"[{name}] head headers:")
    for h in headers:
        print(f"[{name}] >{h}")
    print(f"[{name}] tail headers:")
    for h in last_headers:
        print(f"[{name}] >{h}")
    print("")

In [90]:
for name in split_names:
    inspect_fasta(split_fasta_paths[name], name=name, n=5)

[train] /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_ptns.faa
[train] n_records: 13,942,367
[train] head headers:
[train] >1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_1
[train] >1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_2
[train] >1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_3
[train] >1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_4
[train] >1000562.SAMN03114893.JSAP01000006~37-76653~standalone_mge_5
[train] tail headers:
[train] >fragment_9999_7
[train] >fragment_9999_8
[train] >fragment_9999_9
[train] >fragment_9999_10
[train] >fragment_9999_11

[test_equal_pos] /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_pos_ptns.faa
[test_equal_pos] n_records: 380,925
[test_equal_pos] head headers:
[test_equal_pos] >1001585.SAMN02603190.CP002620~3298858-3398487~standalone_mge_1
[test_equal_pos] >1001585.SAMN02603190.CP002620~3298858-3398487~standalone_mge_2
[test_equal_pos] 

## Prepare CHTC files
The ESM embedding jobs will not be run locally. Instead, they will be run on the [UW-Madison Center for High Throughput Computing](https://chtc.cs.wisc.edu/) to utilize their GPUs. CHTC uses [HTCondor](https://htcondor.org/), not Slurm. The required `inputs.csv` file to run these jobs are generated in this section. The job submit file `job.sub` and executable script `run.sh` are in the folder `./files/esm_embed/`.

### Prepare the `inputs.csv` for generating protein embeddings using the training data chunks

In [91]:
import itertools as it
import os
from pathlib import Path

In [92]:
# train chunk files (in numeric order)
train_faa = sorted(
    [p for p in TRAIN_SPLIT_DIR.glob("train_chunk_*.faa")],
    key=lambda p: int(p.stem.split("_")[-1]),
)

# all test split FASTAs (train excluded), deterministic order
test_faa = sorted(
    [p for p in (TEST_OUT).glob("*_ptns.faa") if p.name != "train_ptns.faa"],
    key=lambda p: p.name,
)

faa_paths = train_faa + test_faa
faa = [p.name for p in faa_paths]

# dataset names aligned 1:1 with faa_paths
output = (
    [f"esm_train_data_{i}" for i in range(len(train_faa))]
    + [f"esm_test_data_{p.stem.replace('_ptns', '')}" for p in test_faa]
)

model_esm = ["esm2_t30_150M"] * len(output)
accelerator = ["gpu"] * len(output)
devices = ["1"] * len(output)

print(f"train chunks: {len(train_faa):,}")
print(f"test splits: {len(test_faa):,}")
print(f"total jobs: {len(faa_paths):,}")

train chunks: 15
test splits: 10
total jobs: 25


In [93]:
# show first few mappings
for i in range(min(10, len(faa_paths))):
    print(f"{i:02d}  {faa_paths[i]}  ->  {output[i]}")

00  /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_split/train_chunk_0.faa  ->  esm_train_data_0
01  /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_split/train_chunk_1.faa  ->  esm_train_data_1
02  /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_split/train_chunk_2.faa  ->  esm_train_data_2
03  /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_split/train_chunk_3.faa  ->  esm_train_data_3
04  /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_split/train_chunk_4.faa  ->  esm_train_data_4
05  /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_split/train_chunk_5.faa  ->  esm_train_data_5
06  /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_split/train_chunk_6.faa  ->  esm_train_data_6
07  /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_split/train_chunk_7.fa

In [94]:
CHTC_FILES_DIR = Path("./files/esm_embed")
CHTC_FILES_DIR.mkdir(exist_ok=True)

In [95]:
with open(os.path.join(CHTC_FILES_DIR, "inputs.csv"), "w") as fp:
    for row in zip(faa, output, model_esm, accelerator, devices):
        fp.write(",".join(row) + "\n")

### Prepare the `job.sub` and `run.sh` files

Need to use CHTC GPUs.

In [56]:
! cat {CHTC_FILES_DIR.joinpath("job.sub")}

universe = vanilla
executable = run.sh
get_env = GRPSTAG, STAGING, USER
arguments = $(faa) $(outdir) $(model_esm) $(accelerator) $(devices)

Requirements = (Target.HasCHTCStaging == true)

require_gpus = (DriverVersion >= 11.6) && (GlobalMemoryMB >= 50000)
request_gpus = 1

+WantGPULab = true

+GPUJobLength = "short" 

request_cpus = 1

request_memory = 8GB
request_disk = 32GB

output = $(Cluster)_$(Process).out
log = $(Cluster)_$(Process).log
error = $(Cluster)_$(Process).err

queue faa,outdir,model_esm,accelerator,devices from inputs.csv

In [57]:
! cat {CHTC_FILES_DIR.joinpath("inputs.csv")}

train_chunk_0.faa,esm_train_data_0,esm2_t30_150M,gpu,1
train_chunk_1.faa,esm_train_data_1,esm2_t30_150M,gpu,1
train_chunk_2.faa,esm_train_data_2,esm2_t30_150M,gpu,1
train_chunk_3.faa,esm_train_data_3,esm2_t30_150M,gpu,1
train_chunk_4.faa,esm_train_data_4,esm2_t30_150M,gpu,1
train_chunk_5.faa,esm_train_data_5,esm2_t30_150M,gpu,1
train_chunk_6.faa,esm_train_data_6,esm2_t30_150M,gpu,1
train_chunk_7.faa,esm_train_data_7,esm2_t30_150M,gpu,1
train_chunk_8.faa,esm_train_data_8,esm2_t30_150M,gpu,1
train_chunk_9.faa,esm_train_data_9,esm2_t30_150M,gpu,1
train_chunk_10.faa,esm_train_data_10,esm2_t30_150M,gpu,1
train_chunk_11.faa,esm_train_data_11,esm2_t30_150M,gpu,1
train_chunk_12.faa,esm_train_data_12,esm2_t30_150M,gpu,1
train_chunk_13.faa,esm_train_data_13,esm2_t30_150M,gpu,1
train_chunk_14.faa,esm_train_data_14,esm2_t30_150M,gpu,1
test_equal_pos_ptns.faa,esm_test_data_test_equal_pos,esm2_t30_150M,gpu,1
test_equal_source_ptns.faa,esm_test_data_test_equal_source,esm2_t30_150M,gpu,1
test_half_vir

In [58]:
! cat {CHTC_FILES_DIR.joinpath("run.sh")}

#!/bin/bash

FAA=$1
OUTDIR=$2
MODELESM=$3
ACCEL=$4
NDEVICES=$5

echo 'Date: ' `date`
echo 'Host: ' `hostname`
echo 'System: ' `uname -spo`
echo 'CUDA_VISIBLE_DEVICES: ' $CUDA_VISIBLE_DEVICES
echo 'Gpu: ' `nvidia-smi -L | grep $CUDA_VISIBLE_DEVICES`

CHECKPOINTSTAR="checkpoints.tar.gz"

cat <<EOF
Train FAA file:           $FAA
Output directory:         $OUTDIR
ESM model:                $MODELESM
Accelerator:              $ACCEL
Number of devices:        $NDEVICES
ESM2 checkpoints archive: $CHECKPOINTSTAR
EOF

set -e
ENVNAME="pst" # conda-packed PST environment name
TARBALL="${ENVNAME}.tar.gz"
ENVDIR=$ENVNAME

# Set DDP debug info
export NCCL_DEBUG="INFO"
export TORCH_CPP_LOG_LEVEL="INFO"
export TORCH_DISTRIBUTED_DEBUG="INFO"
export CUDA_LAUNCH_BLOCKING="1"

# move data

echo "Moving $FAA over from $GRPSTAG/$USER"
cp $GRPSTAG/$USER/$FAA .

echo "Moving $CHECKPOINTSTAR over from $GRPSTAG/$USER"
cp $GRPSTAG/$USER/$CHECKPOINTSTAR .
echo "Untarring $CHECKPOINTSTAR"
tar -xzf $CHECKPOINTSTAR



## Execute jobs
After transfering the `esm_embed` job files and tar-archived folder `checkpoints.tar.gz` containing the pre-trained ESM model checkpoint `esm2_t30_150M` (can be obtained at [github.com/facebookresearch/esm](https://github.com/facebookresearch/esm)).

Here, the ESM embeddings were transferred locally to the location specified below:

In [3]:
ESM_DIR = PST_OUT.joinpath("esm")
ESM_DIR.mkdir(exist_ok=True)

In [7]:
os.system(f'''bash -lc 'shopt -s nullglob; for f in "{ESM_DIR}"/esm_*_data*.tar.gz; do echo "Extracting $f"; tar --use-compress-program="pigz -d -p 16" -xf "$f" -C "{ESM_DIR}" & done; wait; compgen -G "{ESM_DIR}/esm_*_data*.tar.gz" > /dev/null || echo "No matching archives in {ESM_DIR}"' ''')

Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_equal_pos.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_equal_source.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_half_virus_host.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_host_enriched.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_input.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_mge_enriched.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_near_all_host.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_near_all_virus.tar.gz
Extracting /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_tes

0

In [8]:
os.system(f'''bash -lc 'shopt -s nullglob globstar; for f in "{ESM_DIR}"/**/*.h5; do bn=$(basename "$f"); pd=$(basename "$(dirname "$f")"); case "$pd" in esm_train_data_*|esm_test_data_*) pfx="$pd" ;; *) echo "skip $f (parent=$pd)"; continue ;; esac; desired="${{pfx}}_esm2_t30_150M_results.h5"; [ "$bn" = "$desired" ] && continue; new="$(dirname "$f")/$desired"; echo "Renaming: $f -> $new"; mv -n "$f" "$new"; done' ''')

Renaming: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_equal_pos/esm2_t30_150M_results.h5 -> /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_equal_pos/esm_test_data_test_equal_pos_esm2_t30_150M_results.h5
Renaming: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_equal_source/esm2_t30_150M_results.h5 -> /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_equal_source/esm_test_data_test_equal_source_esm2_t30_150M_results.h5
Renaming: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_half_virus_host/esm2_t30_150M_results.h5 -> /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_half_virus_host/esm_test_data_test_half_virus_host_esm2_t30_150M_results.h5
Renaming: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/esm/esm_test_data_test_host_enriched/esm2_t30_150M_results.h5 

0